# Amortized EI with a PFN-VLA action expert: does reading the frozen PFN help?

## Description

**Question.** A policy amortized over the prior has to *read* the PFN's belief. The VLA recipe does this with a small action expert that cross-attends to a frozen backbone's per-layer keys and values (π0). The cleanest test of that architecture is to have it learn a known acquisition function: one forward pass from the context to the distribution of EI's maximizer, with no candidate scoring. **Primary question (A4): does an expert reading `pfn_large`'s context K/V learn amortized EI better than the same expert reading an identically shaped encoder trained from scratch on the same data (D-only)?** If not, the frozen PFN's representation gives the policy nothing that the training data alone doesn't. The predecessor answered no on the small smoke PFN (D-only 0.900 vs 0.889 at d = 6). `pfn_large` is a 6.6 M-parameter model trained 50× longer, so the answer may differ.

**Setup.**
- **States.** BNN-prior tasks at d = 2, split 3 for training (4096 tasks), split 4 for test (256 tasks). From each task:
  - every prefix of an EI trajectory (n_init = 4, T = 20, both run with pfn_large), which is the distribution a BO policy meets;
  - as many random uniform contexts with n ~ U{1, …, 24}.

  That gives 163,840 training and 10,240 test states.
- **Target.** p(x) ∝ exp((EI(x) − max EI) / (0.03 · max EI)) over 4096 shifted-Sobol candidates plus 3 × 64 incumbent perturbations. EI comes from pfn_large. Training uses the 128 heaviest candidates, renormalized. The loss is the forward KL (cross-entropy).
- **Expert.**
  - 8 layers, one per PFN layer, width 128, 4 heads;
  - tokens: an action token (plus the budget left) and the incumbent;
  - head: a 16-component Gaussian mixture in logit space.

  The trunks are:
  - `pfn_all`: layer l reads PFN layer l;
  - `pfn_last`: every layer reads the last PFN layer;
  - `d_only`: reads a PFN with pfn_large's exact architecture, randomly initialized and trained jointly. That is everything the frozen PFN has except its pretraining.
  - `pfn_all` + 256 **probes**: also reads the frozen PFN's final state at 256 fixed Sobol points, i.e. its prediction at query points. This is the predecessor's fix.

  Identical data, steps (20k × 256), optimizer and seed for every trunk.
- **Reference.** EI maximized over 256 random points: a cost-matched alternative to one-pass amortization.

**What to look for.**
- **M1:** the EI ratio EI(x̂)/max EI on test states, for the mixture's mode and for a sample (median and 10th percentile), split by state source (trajectory vs random) and by context size.
- **Closed loop:** 200 fresh tasks (split 5), T = 20. The expert (mode, sample) runs against EI with the full candidate optimizer, compared by the paired geometric-mean regret ratio.

**Decision rule (fixed before the run).**
- **A4 GO** (the frozen representation helps) if the mean paired difference of test-state EI ratios (mode), `pfn_all − d_only`, is ≥ 0.02 with its 95% bootstrap CI above 0.
- **A4 NO-GO** if that CI lies entirely below +0.02, i.e. no advantage worth having.
- **INCONCLUSIVE** otherwise.
- *Secondary (reported, not gating).* `pfn_last` vs `pfn_all` (does depth matter); probes vs `pfn_all`; closed-loop non-inferiority to EI (geometric-mean regret ratio CI upper bound < 1.25).

**Limits.**
- d = 2 only: the predecessor's readout degraded quickly with d, and d = 2 is where a negative result is most informative.
- One seed per trunk. Task-to-task variation is in the CIs; training-seed variation is not. A difference that hinges on it is reported as such.
- In-prior, noise-free.
- The probe trunk reintroduces 256 PFN query tokens per decision, so it is no longer "one pass with no candidate scoring".


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anytime_acq.analysis import load_snapshot, show_header
from anytime_acq.analysis.stats import bootstrap_ci

SNAPSHOT = "results/ei_readout/<sweep_id>"  # set after `just snapshot`
snap = load_snapshot(SNAPSHOT)
show_header(snap)

## Results

### Training

In [ ]:
runs = snap.runs.set_index("member")
label = runs["task.trunk"] + np.where(runs.get("task.n_probe", 0).fillna(0).astype(int) > 0, "+probe", "")
cur = snap.curves().assign(trunk=lambda x: x.member.map(label))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, key in zip(axes, ["train/loss", "val/loss"], strict=True):
    for trunk, g in cur.loc[cur.key == key].groupby("trunk"):
        ax.plot(g.step, g.value.rolling(5 if key.startswith("train") else 1, min_periods=1).mean(), label=trunk)
    ax.set_title(key); ax.set_xlabel("step"); ax.grid(alpha=0.3)
axes[0].legend(); fig.tight_layout()
runs.assign(trunk=label)[["trunk", "model/trainable_params", "final/val_loss", "time/train_s"]]

### One-step quality: EI ratio on test states

EI(x̂) / max EI over the 4288-point reference set; ≥ 1 is possible (the expert can land between candidates). The cost-matched reference is EI maximized over 256 random points.

In [ ]:
r = snap.table("ei_ratio").assign(trunk=lambda x: x.member.map(label))
cheap = snap.table("cheap").groupby("state").ratio.first()
def q(x):
    return pd.Series({"median": np.median(x), "p10": np.quantile(x, 0.1), "mean": np.mean(x)})
tab = pd.concat({f"{t} ({how})": q(g[f"ratio_{how}"]) for t, g in r.groupby("trunk") for how in ("mode", "sample")}, axis=1).T
tab.loc["EI over 256 random points"] = q(cheap)
tab.round(3)

In [ ]:
by = r.assign(n_bin=pd.cut(r.n, [0, 4, 8, 12, 16, 24])).groupby(["trunk", "source", "n_bin"], observed=True).ratio_mode.median().unstack(0)
by.round(3)

### A4: PFN-reading vs D-only (pre-registered)

Paired over test states: the mean of ratio(pfn_all) − ratio(d_only) (mode), with a 95% bootstrap CI.

In [ ]:
w = r.pivot_table(index="state", columns="trunk", values="ratio_mode")
def diff(a, b):
    x = (w[a] - w[b]).dropna().to_numpy()
    lo, hi = bootstrap_ci(x, n_boot=4000)
    return {"a": a, "b": b, "mean_diff": x.mean(), "ci_lo": lo, "ci_hi": hi, "n": len(x)}
comps = [("pfn_all", "d_only"), ("pfn_all", "pfn_last")] + ([("pfn_all+probe", "pfn_all")] if "pfn_all+probe" in w else [])
a4 = pd.DataFrame([diff(a, b) for a, b in comps if a in w and b in w])
a4.round(4)

### Closed loop on fresh tasks

Geometric-mean final-regret ratio to EI (paired over tasks); < 1 = better than EI.

In [ ]:
cl = snap.table("closed_loop").assign(trunk=lambda x: x.member.map(label))
cl["log_regret"] = np.log10((cl.best_f - cl.groupby("seed").fstar.transform("min")).clip(lower=0) + 1e-4)
rows = []
for trunk, g in cl.groupby("trunk"):
    p = g.pivot_table(index="seed", columns="method", values="log_regret")
    for m in ("expert_mode", "expert_sample"):
        x = (p[m] - p["ei"]).to_numpy(); lo, hi = bootstrap_ci(x, n_boot=4000)
        rows.append({"trunk": trunk, "method": m, "ratio": 10 ** x.mean(), "ci_lo": 10**lo, "ci_hi": 10**hi, "non_inferior_1.25": 10**hi < 1.25})
pd.DataFrame(rows).round(3)

## Observations

_Filled from the outputs above._

In [ ]:
row = a4.loc[(a4.a == "pfn_all") & (a4.b == "d_only")].iloc[0]
verdict = "GO" if row.mean_diff >= 0.02 and row.ci_lo > 0 else ("NO-GO" if row.ci_hi < 0.02 else "INCONCLUSIVE")
print(f"A4: pfn_all - d_only = {row.mean_diff:+.4f} [{row.ci_lo:+.4f}, {row.ci_hi:+.4f}] -> {verdict}")

## Verdict

_From the rule above; recorded with `just decide`._